# Data Analysis For All Weather Stations From IMS

In [ ]:
import pandas as pd
from pathlib import Path
from weather_engine import utils as ut

def get_missingness_report(df: pd.DataFrame, file_path: Path):
    """
    Calculates missingness percentage for specific features and returns a summary row.
    """
    feature_cols = df.columns.tolist()
    
    station_name = Path(file_path).stem.split("2020-2026")[0]

    if not df.empty:
        station_id = df['station_id'].iloc[0]
    else:
        station_id = 'Unknown'

    report = {
        'station_name': station_name,
        'station_id': station_id
    }
    
    for col in feature_cols:
        if df.empty:
            report[f"{col}_missingness"] = 100.0
        else:
            missing_pct = (df[col].isna().sum() / len(df)) * 100
            report[f"{col}_missingness"] = round(missing_pct, 2)
            
    return report


data_dir =  ut.get_project_root() / 'data/'
csv_files = list(data_dir.glob('*.csv'))

missingness_results = []

print(f"Found {len(csv_files)} files. Starting cyclic processing...")

for file_path in csv_files:
    current_df = pd.read_csv(file_path)
    
    station_report = get_missingness_report(current_df, file_path)
    missingness_results.append(station_report)
    
    del current_df

final_report_df = pd.DataFrame(missingness_results)
missingness_cols = [c for c in final_report_df.columns if c.endswith('_missingness')]

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    styled_df = final_report_df.style.background_gradient(
        cmap='RdYlGn_r', 
        subset=missingness_cols, 
        vmin=0, 
        vmax=100
    )
    
    display(styled_df)

Found 89 files. Starting cyclic processing...


,station_name,station_id,timestamp_missingness,station_id_missingness,latitude_missingness,longitude_missingness,elevation_missingness,rain_missingness,wsmax_missingness,wdmax_missingness,ws_missingness,wd_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,ws1mm_missingness,ws10mm_missingness
0,MEROM_GOLAN_PICMAN_,10,0.000000,0.000000,0.000000,0.000000,100.000000,0.140000,0.020000,0.020000,0.020000,0.030000,0.020000,0.020000,0.020000,0.020000,0.020000,0.020000,0.020000
1,KEFAR_BLUM_,202,0.000000,0.000000,0.000000,0.000000,100.000000,9.410000,0.010000,0.050000,0.010000,0.050000,0.050000,0.020000,0.060000,0.030000,0.020000,0.010000,0.010000
2,NETIV_HALAMED_HE_,25,0.000000,0.000000,0.000000,0.000000,100.000000,0.270000,0.370000,0.360000,0.370000,0.360000,0.340000,0.060000,0.030000,0.070000,0.100000,0.370000,0.370000
3,HAR_HARASHA_,24,0.000000,0.000000,0.000000,0.000000,100.000000,1.780000,0.030000,0.090000,0.160000,0.090000,0.020000,0.020000,0.020000,0.020000,0.070000,0.020000,0.020000
4,GAT_,236,0.000000,0.000000,0.000000,0.000000,100.000000,3.500000,100.000000,100.000000,100.000000,100.000000,100.000000,0.030000,0.050000,0.040000,0.020000,100.000000,100.000000
5,NEGBA_,82,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.030000,0.380000,0.030000,0.380000,0.380000,0.050000,0.050000,0.030000,0.030000,0.030000,0.030000
6,DOROT_,79,0.000000,0.000000,0.000000,0.000000,100.000000,0.890000,0.020000,0.030000,0.120000,0.030000,0.020000,0.050000,0.070000,0.050000,0.040000,0.020000,41.130000
7,EN_HAHORESH_,107,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,100.000000,100.000000,100.000000,100.000000,100.000000,0.020000,0.020000,0.020000,0.020000,100.000000,100.000000
8,MIZPE_MIDRAG_,504,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
9,YOTVATA_,36,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.020000,0.020000,0.020000,0.020000,0.020000,0.030000,0.020000,0.030000,0.030000,0.020000,17.780000


## Filtering out problematic stations that miss alot of sensors

In [9]:
cols_to_check = [c for c in final_report_df.columns if c.endswith('_missingness')]
cols_to_check.remove('elevation_missingness')

print(cols_to_check)

stations_to_leave = []

for idx, row in final_report_df.iterrows():
    for col in cols_to_check:
        if row[col] >= 30.0:
            stations_to_leave.append(row['station_id'])
            break
            
df_bad_stations = final_report_df[final_report_df['station_id'].isin(stations_to_leave)]
df_bad_stations
            

['timestamp_missingness', 'station_id_missingness', 'latitude_missingness', 'longitude_missingness', 'rain_missingness', 'wsmax_missingness', 'wdmax_missingness', 'ws_missingness', 'wd_missingness', 'stdwd_missingness', 'td_missingness', 'rh_missingness', 'tdmax_missingness', 'tdmin_missingness', 'ws1mm_missingness', 'ws10mm_missingness']


,station_name,station_id,timestamp_missingness,station_id_missingness,latitude_missingness,longitude_missingness,elevation_missingness,rain_missingness,wsmax_missingness,wdmax_missingness,ws_missingness,wd_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,ws1mm_missingness,ws10mm_missingness
4,GAT_,236,0.0,0.0,0.0,0.0,100.0,3.50,100.00,100.00,100.00,100.00,100.00,0.03,0.05,0.04,0.02,100.00,100.00
6,DOROT_,79,0.0,0.0,0.0,0.0,100.0,0.89,0.02,0.03,0.12,0.03,0.02,0.05,0.07,0.05,0.04,0.02,41.13
7,EN_HAHORESH_,107,0.0,0.0,0.0,0.0,100.0,0.00,100.00,100.00,100.00,100.00,100.00,0.02,0.02,0.02,0.02,100.00,100.00
8,MIZPE_MIDRAG_,504,0.0,0.0,0.0,0.0,100.0,0.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
10,TEL_AVIV_COAST_,178,0.0,0.0,0.0,0.0,100.0,0.02,0.38,0.38,0.38,0.38,2.29,0.44,0.41,0.45,0.43,0.38,39.87
21,HAKFAR_HAYAROK_,275,0.0,0.0,0.0,0.0,100.0,5.74,100.00,100.00,100.00,100.00,100.00,1.09,1.13,100.00,100.00,100.00,100.00
23,NAZARETH_,500,0.0,0.0,0.0,0.0,100.0,0.00,5.55,5.55,5.55,5.55,5.55,0.02,0.01,0.01,0.02,5.55,100.00
25,NEWE_ZOHAR_UNI_,32,0.0,0.0,0.0,0.0,100.0,100.00,100.00,100.00,100.00,100.00,100.00,0.18,0.17,100.00,100.00,100.00,100.00
27,AMMIAD_,123,0.0,0.0,0.0,0.0,100.0,0.00,100.00,100.00,100.00,100.00,100.00,1.93,1.93,1.93,1.93,100.00,100.00
28,BEER_SHEVA_BGU_,411,0.0,0.0,0.0,0.0,100.0,0.00,1.87,1.87,1.87,1.87,1.88,0.03,0.03,0.03,0.03,1.88,68.33


In [10]:
# How many stations have ANY column over 30%?
print(f"Total stations: {len(final_report_df)}")
print(f"Stations with 30%+ missingness in any column: {len(stations_to_leave)}")

# Which columns are causing the most flags?
for col in cols_to_check:
    over_30 = (final_report_df[col] >= 30.0).sum()
    if over_30 > 0:
        print(f"{col}: {over_30} stations over 30%")

Total stations: 89
Stations with 30%+ missingness in any column: 35
timestamp_missingness: 1 stations over 30%
station_id_missingness: 1 stations over 30%
latitude_missingness: 1 stations over 30%
longitude_missingness: 1 stations over 30%
rain_missingness: 5 stations over 30%
wsmax_missingness: 22 stations over 30%
wdmax_missingness: 22 stations over 30%
ws_missingness: 22 stations over 30%
wd_missingness: 22 stations over 30%
stdwd_missingness: 22 stations over 30%
td_missingness: 4 stations over 30%
rh_missingness: 4 stations over 30%
tdmax_missingness: 6 stations over 30%
tdmin_missingness: 6 stations over 30%
ws1mm_missingness: 23 stations over 30%
ws10mm_missingness: 34 stations over 30%
